In [ ]:


!pip install xgboost
!pip install spacy


# download spacy model
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import numpy as np
import pandas as pd
import nltk
import spacy

In [ ]:
nlp = spacy.load("en_core_web_sm")

print("spaCy loaded successfully")

spaCy loaded successfully


In [ ]:
import re



In [ ]:
# Tokenization
from nltk.tokenize import word_tokenize, sent_tokenize

# Stopwords
from nltk.corpus import stopwords

In [ ]:

# Lemmatization
from nltk.stem import WordNetLemmatizer

# POS tagging
from nltk import pos_tag

# WordNet
from nltk.corpus import wordnet

In [ ]:
# Machine learning utilities
from sklearn.model_selection import train_test_split

In [ ]:
# Feature extraction
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
# Evaluation
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

# Model saving
import pickle


In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [ ]:
!pip install kagglehub[pandas-datasets]

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "CUAD_v1/master_clauses.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "konradb/atticus-open-contract-dataset-aok-beta",
    file_path
)

print("Dataset shape:", df.shape)
print(df.head())

/tmp/ipykernel_20608/1107126810.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'atticus-open-contract-dataset-aok-beta' dataset.
Dataset shape: (510, 83)
                                            Filename  \
0  CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...   
1  EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...   
2  FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...   
3  GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10...   
4  IdeanomicsInc_20160330_10-K_EX-10.26_9512211_E...   

                                    Document Name  \
0               ['MARKETING AFFILIATE AGREEMENT']   
1   ['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT']   
2  ['CONTENT DISTRIBUTION AND LICENSE AGREEMENT']   
3           ['WEBSITE CONTENT LICENSE AGREEMENT']   
4                   ['CONTENT LICENSE AGREEMENT']   

                         Document Name-Answer  \
0               MARKETING AFFILIATE AGREEMENT   
1   VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT   
2  CONTENT DISTRIBUTION AND LICENSE AGREEMENT   
3           WEBSITE CONTENT LI

In [ ]:
df.columns

Index(['Filename', 'Document Name', 'Document Name-Answer', 'Parties',
       'Parties-Answer', 'Agreement Date', 'Agreement Date-Answer',
       'Effective Date', 'Effective Date-Answer', 'Expiration Date',
       'Expiration Date-Answer', 'Renewal Term', 'Renewal Term-Answer',
       'Notice Period To Terminate Renewal',
       'Notice Period To Terminate Renewal- Answer', 'Governing Law',
       'Governing Law-Answer', 'Most Favored Nation',
       'Most Favored Nation-Answer', 'Competitive Restriction Exception',
       'Competitive Restriction Exception-Answer', 'Non-Compete',
       'Non-Compete-Answer', 'Exclusivity', 'Exclusivity-Answer',
       'No-Solicit Of Customers', 'No-Solicit Of Customers-Answer',
       'No-Solicit Of Employees', 'No-Solicit Of Employees-Answer',
       'Non-Disparagement', 'Non-Disparagement-Answer',
       'Termination For Convenience', 'Termination For Convenience-Answer',
       'Rofr/Rofo/Rofn', 'Rofr/Rofo/Rofn-Answer', 'Change Of Control',
      

In [ ]:
clause_columns = [
    c for c in df.columns
    if "-Answer" not in c
    and "- Answer" not in c
    and c not in ["Filename", "Document Name"]
]

print("Number of clause columns:", len(clause_columns))
print(clause_columns[:20])

Number of clause columns: 40
['Parties', 'Agreement Date', 'Effective Date', 'Expiration Date', 'Renewal Term', 'Notice Period To Terminate Renewal', 'Governing Law', 'Most Favored Nation', 'Competitive Restriction Exception', 'Non-Compete', 'Exclusivity', 'No-Solicit Of Customers', 'No-Solicit Of Employees', 'Non-Disparagement', 'Termination For Convenience', 'Rofr/Rofo/Rofn', 'Change Of Control', 'Anti-Assignment', 'Revenue/Profit Sharing', 'Price Restrictions']


In [ ]:
rows = []

for _, row in df.iterrows():
    for clause in clause_columns:
        text = row[clause]

        if pd.notna(text) and str(text).strip() != "":
            rows.append({
                "clause_text": text,
                "clause_type": clause
            })

clauses_df = pd.DataFrame(rows)

print("New dataset shape:", clauses_df.shape)
print(clauses_df.head())

New dataset shape: (20400, 2)
                                         clause_text      clause_type
0  ['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...          Parties
1             ['8th day of May 2014', 'May 8, 2014']   Agreement Date
2  ['This agreement shall begin upon the date of ...   Effective Date
3  ['This agreement shall begin upon the date of ...  Expiration Date
4  ['This agreement shall begin upon the date of ...     Renewal Term


In [ ]:
clauses_df["clause_text"] = clauses_df["clause_text"].astype(str)

In [ ]:
clauses_df["clause_text"] = clauses_df["clause_text"].str.replace(r"[\[\]']", "", regex=True)

In [ ]:
clauses_df = clauses_df[clauses_df["clause_text"].str.strip() != ""]

In [ ]:
print(clauses_df.shape)
clauses_df.head()

(6192, 2)


,clause_text,clause_type
0,"BIRCH FIRST GLOBAL INVESTMENTS INC., MA, Marke...",Parties
1,"8th day of May 2014, May 8, 2014",Agreement Date
2,This agreement shall begin upon the date of it...,Effective Date
3,This agreement shall begin upon the date of it...,Expiration Date
4,This agreement shall begin upon the date of it...,Renewal Term


In [ ]:
data = clauses_df[["clause_text", "clause_type"]].copy()

print(data.shape)
data.head()

(6192, 2)


,clause_text,clause_type
0,"BIRCH FIRST GLOBAL INVESTMENTS INC., MA, Marke...",Parties
1,"8th day of May 2014, May 8, 2014",Agreement Date
2,This agreement shall begin upon the date of it...,Effective Date
3,This agreement shall begin upon the date of it...,Expiration Date
4,This agreement shall begin upon the date of it...,Renewal Term


In [ ]:
stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

In [ ]:
import string

def preprocess_text(text):

    # lowercase
    text = text.lower()

    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords
    tokens = [w for w in tokens if w not in stop_words]

    # lemmatization
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return " ".join(tokens)

In [ ]:
data["clean_text"] = data["clause_text"].apply(preprocess_text)

In [ ]:
data.head()

,clause_text,clause_type,clean_text
0,"BIRCH FIRST GLOBAL INVESTMENTS INC., MA, Marke...",Parties,birch first global investment inc marketing af...
1,"8th day of May 2014, May 8, 2014",Agreement Date,8th day may 2014 may 8 2014
2,This agreement shall begin upon the date of it...,Effective Date,agreement shall begin upon date execution acce...
3,This agreement shall begin upon the date of it...,Expiration Date,agreement shall begin upon date execution acce...
4,This agreement shall begin upon the date of it...,Renewal Term,agreement shall begin upon date execution acce...


In [ ]:
risky_clauses = [
    "Non-Compete",
    "Exclusivity",
    "Termination For Convenience",
    "Anti-Assignment",
    "Uncapped Liability",
    "Cap On Liability",
    "Liquidated Damages",
    "Minimum Commitment",
    "Volume Restriction",
    "Price Restrictions"
]

In [ ]:
data["risk"] = data["clause_type"].apply(
    lambda x: 1 if x in risky_clauses else 0
)

In [ ]:
data.head()

,clause_text,clause_type,clean_text,risk
0,"BIRCH FIRST GLOBAL INVESTMENTS INC., MA, Marke...",Parties,birch first global investment inc marketing af...,0
1,"8th day of May 2014, May 8, 2014",Agreement Date,8th day may 2014 may 8 2014,0
2,This agreement shall begin upon the date of it...,Effective Date,agreement shall begin upon date execution acce...,0
3,This agreement shall begin upon the date of it...,Expiration Date,agreement shall begin upon date execution acce...,0
4,This agreement shall begin upon the date of it...,Renewal Term,agreement shall begin upon date execution acce...,0


In [ ]:
data["risk"].value_counts()

,count
risk,
0,4627
1,1565


In [ ]:
X = data["clean_text"]
y = data["risk"]

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)

In [ ]:
X_tfidf = tfidf.fit_transform(X)

In [ ]:
print(X_tfidf.shape)

(6192, 5000)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

(4953, 5000)
(1239, 5000)


In [ ]:
model_lr = LogisticRegression()

model_lr.fit(X_train, y_train)

LogisticRegression()

In [ ]:
y_pred = model_lr.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.8894269572235673
              precision    recall  f1-score   support

           0       0.87      0.99      0.93       886
           1       0.95      0.64      0.77       353

    accuracy                           0.89      1239
   macro avg       0.91      0.82      0.85      1239
weighted avg       0.90      0.89      0.88      1239



In [ ]:
model_nb = MultinomialNB()

model_nb.fit(X_train, y_train)

MultinomialNB()

In [ ]:
y_pred_nb = model_nb.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.8555286521388217
              precision    recall  f1-score   support

           0       0.84      0.99      0.91       886
           1       0.94      0.52      0.67       353

    accuracy                           0.86      1239
   macro avg       0.89      0.76      0.79      1239
weighted avg       0.87      0.86      0.84      1239



In [ ]:
model_rf = RandomForestClassifier(n_estimators=100)

model_rf.fit(X_train, y_train)

RandomForestClassifier()

In [ ]:
y_pred_rf = model_rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.8861985472154964
              precision    recall  f1-score   support

           0       0.88      0.98      0.92       886
           1       0.91      0.66      0.77       353

    accuracy                           0.89      1239
   macro avg       0.90      0.82      0.85      1239
weighted avg       0.89      0.89      0.88      1239



In [ ]:
model_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

model_xgb.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [05:36:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred_xgb = model_xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

print(classification_report(y_test, y_pred_xgb))

XGBoost Accuracy: 0.9055690072639225
              precision    recall  f1-score   support

           0       0.91      0.96      0.94       886
           1       0.89      0.76      0.82       353

    accuracy                           0.91      1239
   macro avg       0.90      0.86      0.88      1239
weighted avg       0.90      0.91      0.90      1239



In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
import heapq

In [ ]:
def summarize_contract_cooccur(text, num_sentences=3):

    sentences = sent_tokenize(text)

    words = [
        w.lower() for w in word_tokenize(text)
        if w.lower() not in stop_words and w.isalpha()
    ]

    # word frequency
    word_freq = {}

    for w in words:
        word_freq[w] = word_freq.get(w, 0) + 1

    # co-occurrence matrix
    co_matrix = build_cooccurrence(words, window_size=2)

    # sentence scoring
    sentence_scores = {}

    for sent in sentences:

        tokens = word_tokenize(sent.lower())

        score = 0

        for i in range(len(tokens)-1):

            w1 = tokens[i]
            w2 = tokens[i+1]

            if w1 in word_freq:
                score += word_freq[w1]

            if (w1, w2) in co_matrix:
                score += co_matrix[(w1, w2)]

        sentence_scores[sent] = score

    summary = heapq.nlargest(num_sentences, sentence_scores, key=sentence_scores.get)

    return " ".join(summary)

In [ ]:
from collections import defaultdict

def build_cooccurrence(tokens, window_size=2):

    co_matrix = defaultdict(int)

    for i in range(len(tokens)):
        for j in range(i+1, min(i+window_size, len(tokens))):

            pair = (tokens[i], tokens[j])
            co_matrix[pair] += 1

    return co_matrix

In [ ]:
contract_text = """
This agreement shall begin on May 8 2014.
The company may terminate the agreement without notice.
Payment shall be made within 30 days of invoice.
The agreement will automatically renew unless terminated.
"""

summary = summarize_contract_cooccur(contract_text)

print(summary)

The company may terminate the agreement without notice. 
This agreement shall begin on May 8 2014. The agreement will automatically renew unless terminated.


In [ ]:

def detect_risk(text):

    clean = preprocess_text(text)

    vec = tfidf.transform([clean])

    prediction = model_xgb.predict(vec)[0]

    if prediction == 1:
        return "Risky Clause"
    else:
        return "Safe Clause"

In [ ]:
def analyze_contract(contract_text):
    try:
        if contract_text.strip() == "":
            return "Please enter contract text."

        output = ""

        # Summary
        summary = summarize_contract_cooccur(contract_text)
        output += "----- SUMMARY -----\n"
        output += str(summary) + "\n\n"

        # Risk Analysis
        sentences = sent_tokenize(contract_text)

        output += "----- RISK ANALYSIS -----\n"

        found = False

        for sent in sentences:
            result = detect_risk(sent)

            # DEBUG PRINT
            print("Sentence:", sent)
            print("Result:", result)


            if result == "Risky Clause":
                output += "(Risky) " + sent + "\n"
                found = True

        if not found:
            output += "No risky clauses detected."

        return output

    except Exception as e:
        return f"Error occurred:\n{str(e)}"

In [ ]:

path = kagglehub.dataset_download(
    "konradb/atticus-open-contract-dataset-aok-beta"
)

print("Dataset downloaded to:", path)

Using Colab cache for faster access to the 'atticus-open-contract-dataset-aok-beta' dataset.
Dataset downloaded to: /kaggle/input/atticus-open-contract-dataset-aok-beta


In [ ]:
import os
print(os.listdir(path))

['CUAD_v1']


In [ ]:
!cp -r {path}/* /content/

In [ ]:
!pip install gradio

In [ ]:
pickle.dump(model_xgb, open("model.pkl", "wb"))

In [ ]:
pickle.dump(tfidf, open("vectorizer.pkl", "wb"))

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=analyze_contract,
    inputs=gr.Textbox(lines=15, placeholder="Enter contract text here..."),
    outputs=gr.Markdown(),
    title="Contract Risk Analyzer",
    description="Enter contract text to analyze."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://593266cb4493c47520.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
